# Reducción gravimétrica – Facatativá

**Taller 2 de Geofísica – Gravimetría** · Universidad Distrital Francisco José de Caldas
Autores: Samuel Leandro Calderón Pachón (20221025019) · Daniel Ernesto Chicaeme Sánchez (20222025077)
Profesor: Andrés Cárdenas Contreras

Este cuaderno reproduce toda la reducción de los datos de campo, paso a paso, a partir de los Excel originales del profesor.

**Cómo usarlo**
1. Abrir este cuaderno en VS Code (extensión *Jupyter*) y elegir el kernel de Python (arriba a la derecha).
2. Ejecutar las celdas **en orden**, de arriba hacia abajo (Shift + Enter o *Run All*).
3. La primera celda instala las librerías; solo hace falta una vez por computador.

**Estructura esperada del repositorio**
```
datos/originales/   ← los 4 Excel del profesor (sin modificar)
datos/procesados/   ← los Excel que genera este cuaderno
scripts/            ← este cuaderno
```

## 0. Instalación de librerías (una sola vez por computador)

In [1]:
%pip install pyproj openpyxl numpy pandas
%pip install tidegravity --no-deps

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Daniel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Daniel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Configuración general
Ubica la carpeta del repositorio, define las rutas y los parámetros comunes del proyecto.

In [2]:
import datetime
from pathlib import Path

import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pyproj import Transformer
import tidegravity as tg

# Busca la raíz del repositorio (la carpeta que contiene datos/originales)
raiz = Path.cwd()
while not (raiz / "datos" / "originales").exists():
    if raiz.parent == raiz:
        raise SystemExit("No encontré la carpeta datos/originales. Abre el cuaderno dentro del repositorio.")
    raiz = raiz.parent

ORIG = raiz / "datos" / "originales"
PROC = raiz / "datos" / "procesados"
PROC.mkdir(parents=True, exist_ok=True)

ARCHIVO_CAMPO  = ORIG / "DATOS GRAVIMETRICOS CAMPO.xlsx"
ARCHIVO_AMARRE = ORIG / "AMARRE BASE.xlsx"
PASO1 = PROC / "Paso1_Datos_Organizados_Facatativa.xlsx"
PASO2 = PROC / "Paso2_Coordenadas_Facatativa.xlsx"
PASO45 = PROC / "Paso4y5_Mareas_Facatativa.xlsx"

# Parámetros del proyecto (indicados por el profesor, clase 28 oct 2026)
CONST_GRAV1 = 0.0959                      # ciclos 1 a 7
CONST_GRAV2 = 0.0433                      # ciclos 8 a 11
FECHA_GRAV1 = datetime.datetime(2010, 8, 10)
FECHA_GRAV2 = datetime.datetime(2010, 8, 11)
UTC_OFFSET = 5                            # hora Colombia = UTC − 5
EPSG_SALIDA = "EPSG:3116"                 # MAGNA-SIRGAS / Colombia Bogotá zone (Gauss-Krüger origen Bogotá)
BASE = "GPS-3"

# Estilos para los Excel
FUENTE = "Arial"
f = lambda **k: Font(name=FUENTE, size=10, **k)
ENC_FONT = Font(name=FUENTE, size=10, bold=True, color="FFFFFF")
ENC_FILL = PatternFill("solid", start_color="1F3864")
AMARILLO = PatternFill("solid", start_color="FFFF00")
AZUL = PatternFill("solid", start_color="DDEBF7")
NARANJA = PatternFill("solid", start_color="FCE4D6")
BORDE = Border(*(Side(style="thin", color="BFBFBF"),) * 4)

def encabezado(celda, texto):
    celda.value, celda.font, celda.fill, celda.border = texto, ENC_FONT, ENC_FILL, BORDE
    celda.alignment = Alignment(wrap_text=True, horizontal="center", vertical="center")

print("Repositorio:", raiz)

Repositorio: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa


## Paso 1. Organizar y depurar los datos

1. Se lee `DATOS GRAVIMETRICOS CAMPO.xlsx` y se separan los **ciclos**: cada ciclo abre y cierra en la base GPS-3.
2. A cada fila se le da un **ID único** (`Cxx-Estación`), porque los nombres de estación se repiten entre ciclos pero son puntos distintos.
3. Se aplican las **reglas del profesor** (clase 28 oct 2026):
   - **Regla 1:** si una estación tiene una hora inferior a la esperada dentro del ciclo, solamente se cambia de posición (se ordena por hora).
   - **Regla 2:** si una estación tiene una hora inferior a la hora de inicio en la base, se elimina.
4. Se genera `Paso1_Datos_Organizados_Facatativa.xlsx` con una hoja por ciclo, una hoja **Resumen** y una hoja **Leyenda**.

In [3]:
# --- Leer los ciclos del archivo de campo ---
hoja = openpyxl.load_workbook(ARCHIVO_CAMPO).active
ciclos, actual = [], None
for fila, r in enumerate(hoja.iter_rows(values_only=True), start=1):
    if r[0] is None or str(r[0]).lower().startswith("estac"):
        continue                                   # filas vacías o de encabezado
    dato = dict(fila=fila, est=r[0], lat=r[1], lon=r[2], h=r[3], t=r[4], c=r[5], L=r[6])
    if r[0] == BASE:
        if actual is None:
            actual = [dato]                        # apertura del ciclo
        else:
            actual.append(dato); ciclos.append(actual); actual = None   # cierre del ciclo
    elif actual is not None:
        actual.append(dato)
print("Ciclos encontrados:", len(ciclos))

# --- Reglas del profesor ---
movidas, eliminadas = [], []
for n, ciclo in enumerate(ciclos, start=1):
    apertura, cierre, estaciones = ciclo[0], ciclo[-1], ciclo[1:-1]
    # Regla 2: eliminar estaciones con hora menor a la de apertura en la base
    for d in estaciones:
        if d["t"] < apertura["t"]:
            eliminadas.append((n, d["est"], d["t"]))
    estaciones = [d for d in estaciones if d["t"] >= apertura["t"]]
    # Regla 1: ordenar por hora; registrar las estaciones que cambian de posición
    #          (una estación está fuera de lugar si su hora es menor que la de alguna estación anterior)
    hora_max = apertura["t"]
    for d in estaciones:
        if d["t"] < hora_max:
            movidas.append((n, d["est"], d["t"]))
        hora_max = max(hora_max, d["t"])
    ordenadas = sorted(estaciones, key=lambda d: d["t"])
    ciclos[n - 1] = [apertura] + ordenadas + [cierre]

print("Regla 1 – estaciones reordenadas:", movidas or "ninguna")
print("Regla 2 – estaciones eliminadas:", eliminadas or "ninguna")

# --- Construir el Excel del paso 1 ---
wb = openpyxl.Workbook()
ley = wb.active; ley.title = "Leyenda"
lineas = [
    ["Paso 1 – Organización y depuración de datos gravimétricos"],
    ["Proyecto: Determinación de la Anomalía de Bouguer Total en una zona del Municipio de Facatativá"],
    ["Fuente: DATOS GRAVIMETRICOS CAMPO.xlsx (Universidad Nacional – Universidad Distrital, 2010)"],
    [],
    ["Parámetros"],
    ["Constante gravímetro 1", CONST_GRAV1, "Fecha gravímetro 1", FECHA_GRAV1.date(), "Ciclos 1 a 7"],
    ["Constante gravímetro 2", CONST_GRAV2, "Fecha gravímetro 2", FECHA_GRAV2.date(), "Ciclos 8 a 11"],
    [],
    ["Reglas de depuración (clase 28 oct 2026)"],
    ["Regla 1", "Hora inferior a la esperada dentro del ciclo → solo se cambia de posición."],
    ["", "Resultado: " + ("; ".join(f"ciclo {n}, {e} (hora {t:.2f})" for n, e, t in movidas) or "ninguna")],
    ["Regla 2", "Hora inferior a la hora de inicio en la base → se elimina."],
    ["", "Resultado: " + ("; ".join(f"ciclo {n}, {e} (hora {t:.2f})" for n, e, t in eliminadas) or "ninguna estación cumple la condición")],
]
for l in lineas:
    ley.append(l)
for fila in ley.iter_rows():
    for c in fila:
        c.font = f()
ley["A1"].font = Font(name=FUENTE, size=14, bold=True)
for ref in ("B6", "B7", "D6", "D7"):
    ley[ref].fill, ley[ref].font = AMARILLO, f(color="0000FF")
ley["D6"].number_format = ley["D7"].number_format = "dd/mm/yyyy"
for col, ancho in zip("ABCDE", (24, 14, 20, 14, 60)):
    ley.column_dimensions[col].width = ancho

columnas = ["Orden", "ID", "Estación", "Latitud (°)", "Longitud original (°)", "Longitud (° W)",
            "Altura elipsoidal (m)", "Hora decimal (h)", "Hora (hh:mm)", "Constante", "Lectura dial",
            "Revisión regla 1 (orden)", "Revisión regla 2 (inicio base)", "Fila en archivo original", "Observación"]
anchos = [7, 14, 10, 12, 14, 13, 13, 12, 11, 10, 11, 16, 18, 12, 48]
rangos = []
for n, ciclo in enumerate(ciclos, start=1):
    ws = wb.create_sheet(f"Ciclo {n}")
    grav1 = n <= 7
    ws["A1"] = f"Ciclo {n}"; ws["A1"].font = Font(name=FUENTE, size=13, bold=True)
    ws["A2"], ws["B2"] = "Fecha", "=Leyenda!D6" if grav1 else "=Leyenda!D7"
    ws["A3"], ws["B3"] = "Constante", "=Leyenda!B6" if grav1 else "=Leyenda!B7"
    ws["D2"], ws["E2"] = "Base", BASE
    ws["D3"], ws["E3"] = "Gravímetro", 1 if grav1 else 2
    ws["B2"].number_format = "dd/mm/yyyy"
    for j, texto in enumerate(columnas, start=1):
        encabezado(ws.cell(5, j), texto)
        ws.column_dimensions[get_column_letter(j)].width = anchos[j - 1]
    ws.row_dimensions[5].height = 42
    ini = 6; fin = ini + len(ciclo) - 1
    for k, d in enumerate(ciclo):
        r = ini + k
        es_base = k in (0, len(ciclo) - 1)
        obs = ("Apertura" if k == 0 else "Cierre") + f" del ciclo en la base {BASE}" if es_base else ""
        if (n, d["est"], d["t"]) in movidas and not es_base:
            obs = "Regla 1: hora menor a la esperada; se cambió de posición"
        valores = [k + 1, f"C{n:02d}-{d['est']}", d["est"], d["lat"], d["lon"], f"=-E{r}", d["h"], d["t"],
                   f'=TEXT(H{r}/24,"hh:mm")', "=$B$3", d["L"],
                   "OK" if k == 0 else f'=IF(H{r}>=H{r-1},"OK","Fuera de orden")',
                   f'=IF(H{r}>=$H${ini},"OK","Eliminar")', d["fila"], obs]
        for j, v in enumerate(valores, start=1):
            c = ws.cell(r, j, v); c.font = f(); c.border = BORDE
            c.number_format = {4: "0.000000", 5: "0.000000", 6: "0.000000", 7: "0.000", 8: "0.000", 10: "0.0000", 11: "0.0"}.get(j, "General")
            if es_base: c.fill = AZUL
            if obs.startswith("Regla"): c.fill = NARANJA
    ws.freeze_panes = "C6"
    ws.cell(fin + 2, 1, "Estaciones (sin base)").font = f(bold=True); ws.cell(fin + 2, 4, f"=COUNTA(C{ini+1}:C{fin-1})")
    ws.cell(fin + 3, 1, "Filas fuera de orden").font = f(bold=True); ws.cell(fin + 3, 4, f'=COUNTIF(L{ini}:L{fin},"Fuera de orden")')
    ws.cell(fin + 4, 1, "Filas a eliminar").font = f(bold=True); ws.cell(fin + 4, 4, f'=COUNTIF(M{ini}:M{fin},"Eliminar")')
    rangos.append((ini, fin))

res = wb.create_sheet("Resumen", 1)
res["A1"] = "Resumen de ciclos – Paso 1"; res["A1"].font = Font(name=FUENTE, size=13, bold=True)
cols_res = ["Ciclo", "Fecha", "Gravímetro", "Constante", "Hora apertura (h)", "Hora cierre (h)", "Duración (h)",
            "N.º estaciones", "Lectura apertura GPS-3", "Lectura cierre GPS-3", "Diferencia de cierre (dial)",
            "Fuera de orden", "A eliminar"]
for j, texto in enumerate(cols_res, start=1):
    encabezado(res.cell(3, j), texto); res.column_dimensions[get_column_letter(j)].width = 13
for n, (ini, fin) in enumerate(rangos, start=1):
    r, q = 3 + n, f"'Ciclo {n}'!"
    for j, v in enumerate([n, f"={q}B2", f"={q}E3", f"={q}B3", f"={q}H{ini}", f"={q}H{fin}", f"=F{r}-E{r}",
                           f"={q}D{fin+2}", f"={q}K{ini}", f"={q}K{fin}", f"=J{r}-I{r}",
                           f"={q}D{fin+3}", f"={q}D{fin+4}"], start=1):
        c = res.cell(r, j, v); c.font = f(); c.border = BORDE
        c.number_format = {2: "dd/mm/yyyy", 5: "0.000", 6: "0.000", 7: "0.000"}.get(j, "General")

wb.save(PASO1)
print("Guardado:", PASO1)

Ciclos encontrados: 11
Regla 1 – estaciones reordenadas: [(2, 'SP526', 12.2)]
Regla 2 – estaciones eliminadas: ninguna
Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso1_Datos_Organizados_Facatativa.xlsx


## Paso 2. Coordenadas MAGNA-SIRGAS Gauss-Krüger origen Bogotá

Las coordenadas geográficas (WGS84) de cada estación se transforman a coordenadas planas **MAGNA-SIRGAS / Colombia Bogotá zone (EPSG:3116)**, que es el sistema que pide el taller para todos los mapas.
La longitud del archivo original viene en positivo; se usa en negativo porque es longitud Oeste.

**Comprobación:** GPS-3 debe dar Este ≈ 974495.6 m y Norte ≈ 1025740.3 m.

In [4]:
transformador = Transformer.from_crs("EPSG:4326", EPSG_SALIDA, always_xy=True)
wb = openpyxl.load_workbook(PASO1)

for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    encabezado(ws["P5"], "Este (m)"); encabezado(ws["Q5"], "Norte (m)")
    ws.column_dimensions["P"].width = ws.column_dimensions["Q"].width = 14
    fila = 6
    while ws[f"D{fila}"].value is not None:
        lat, lon = ws[f"D{fila}"].value, -ws[f"E{fila}"].value
        este, norte = transformador.transform(lon, lat)
        ws[f"P{fila}"], ws[f"Q{fila}"] = round(este, 3), round(norte, 3)
        fila += 1

wb.save(PASO2)
gps3 = wb["Ciclo 1"]
print(f"GPS-3 → Este = {gps3['P6'].value} m, Norte = {gps3['Q6'].value} m")
print("Guardado:", PASO2)

GPS-3 → Este = 974495.614 m, Norte = 1025740.326 m
Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso2_Coordenadas_Facatativa.xlsx


## Paso 3. Amarre a la base (gravedad absoluta de GPS-3)

B24NW1 es un punto de gravedad conocida (**977389.51 mGal**). Con cada gravímetro se midió varias veces de ida y vuelta entre B24NW1 y GPS-3 (6 lecturas → 5 diferencias).

1. Cada lectura se multiplica por la constante de su gravímetro.
2. Se calcula la diferencia promedio B24NW1 − GPS-3 con cada gravímetro y se promedian los dos.
3. g(GPS-3) = g(B24NW1) − diferencia promedio.

El Excel del profesor da 977384.730 mGal porque una de sus celdas tiene la diferencia escrita a mano; la diferencia con este cálculo es de 0.003 mGal.

In [5]:
G_B24NW1 = 977389.51   # mGal, valor conocido (AMARRE BASE.xlsx)

amarre = openpyxl.load_workbook(ARCHIVO_AMARRE)
# Lecturas de dial en las filas 9 a 14 de cada hoja
lect_g1 = [amarre.worksheets[0][f"I{r}"].value for r in range(9, 15)]
lect_g2 = [amarre.worksheets[1][f"K{r}"].value for r in range(9, 15)]

def diferencia_promedio(lecturas, constante):
    g = np.array(lecturas) * constante
    return np.mean(np.abs(np.diff(g)))         # |B24NW1 − GPS-3| entre lecturas consecutivas

d1 = diferencia_promedio(lect_g1, CONST_GRAV1)
d2 = diferencia_promedio(lect_g2, CONST_GRAV2)
d_prom = (d1 + d2) / 2
G_BASE = G_B24NW1 - d_prom

print(f"Diferencia gravímetro 0.0959: {d1:.3f} mGal")
print(f"Diferencia gravímetro 0.0433: {d2:.3f} mGal")
print(f"Diferencia promedio:          {d_prom:.3f} mGal")
print(f"g(GPS-3) = {G_B24NW1} − {d_prom:.3f} = {G_BASE:.3f} mGal")

Diferencia gravímetro 0.0959: 4.762 mGal
Diferencia gravímetro 0.0433: 4.803 mGal
Diferencia promedio:          4.783 mGal
g(GPS-3) = 977389.51 − 4.783 = 977384.727 mGal


## Pasos 4 y 5. Lecturas en mGal y corrección por mareas

**Paso 4:** lectura (mGal) = lectura del dial × constante del gravímetro.

**Paso 5:** el efecto de marea terrestre se calcula con el método de **Longman (1959)** (librería *Tidegravity*), con la latitud, longitud, altura, fecha y hora UTC de cada lectura.
La corrección es el efecto con signo contrario: **corrección = − efecto de marea**.

- Ciclos 1 a 7 (gravímetro 0.0959): 10 de agosto de 2010
- Ciclos 8 a 11 (gravímetro 0.0433): 11 de agosto de 2010
- Hora UTC = hora local + 5 h

**Comprobación (ciclo 3, GPS-3):** lectura 103.6967 mGal, efecto 0.1473 mGal, corrección −0.1473 mGal.

In [6]:
wb = openpyxl.load_workbook(PASO2)

for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    ciclo = int(ws.title.split()[1])
    fecha = FECHA_GRAV1 if ciclo <= 7 else FECHA_GRAV2
    for col, texto in (("R", "Lectura (mGal)"), ("S", "Hora UTC"),
                       ("T", "Efecto de marea (mGal)"), ("U", "Corrección por marea (mGal)")):
        encabezado(ws[f"{col}5"], texto); ws.column_dimensions[col].width = 14
    fila = 6
    while ws[f"D{fila}"].value is not None:
        lat, lon = ws[f"D{fila}"].value, -ws[f"E{fila}"].value
        alt, hora = ws[f"G{fila}"].value, ws[f"H{fila}"].value
        t_utc = fecha + datetime.timedelta(hours=hora + UTC_OFFSET)
        gm, gs, g0 = tg.solve_longman_tide_scalar(lat, lon, alt, t_utc)
        ws[f"R{fila}"] = f"=K{fila}*J{fila}"          # Paso 4
        ws[f"S{fila}"] = t_utc.strftime("%Y-%m-%d %H:%M")
        ws[f"T{fila}"] = round(float(g0), 4)           # efecto de marea (Longman)
        ws[f"U{fila}"] = f"=-T{fila}"                  # Paso 5: corrección
        for col in "RTU":
            ws[f"{col}{fila}"].number_format = "0.0000"
        fila += 1

wb.save(PASO45)
c3 = wb["Ciclo 3"]
print(f"Ciclo 3, GPS-3 → efecto de marea = {c3['T6'].value} mGal, corrección = {-c3['T6'].value} mGal")
print("Guardado:", PASO45)

Ciclo 3, GPS-3 → efecto de marea = 0.1473 mGal, corrección = -0.1473 mGal
Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso4y5_Mareas_Facatativa.xlsx
